In [1]:
!pip install torch-levenberg-marquardt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.5/963.5 kB 53.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink

# shulman :

In [6]:
import time
import torch
import numpy as np
import pandas as pd
import torch_levenberg_marquardt as tlm
from torch.utils.data import TensorDataset
from bokeh.plotting import figure, show, output_notebook

# Set high matmul precision
torch.set_float32_matmul_precision('high')

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === Load your data from CSV ===
df = pd.read_csv("/content/magnesio-anthophyllite.csv")  # <-- update with your filename
T_data = torch.tensor(df["T"].values, dtype=torch.float32).unsqueeze(1).to(device)
Cp_data = torch.tensor(df["Cp"].values, dtype=torch.float32).unsqueeze(1).to(device)

# Create dataset and dataloader
dataset = TensorDataset(T_data, Cp_data)
loader = tlm.utils.FastDataLoader(dataset, batch_size=512, repeat=10, shuffle=True, device=device)

# === Define Cp model as a neural net with 6 trainable params ===
class CpModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.a = torch.nn.Parameter(torch.tensor([7.73e-3], dtype=torch.float32))
        self.b = torch.nn.Parameter(torch.tensor([1.263e-3], dtype=torch.float32))
        self.c = torch.nn.Parameter(torch.tensor([3.0], dtype=torch.float32))
        self.d = torch.nn.Parameter(torch.tensor([8.47e-3], dtype=torch.float32))
        self.e = torch.nn.Parameter(torch.tensor([2.0825e-7], dtype=torch.float32))
        self.f = torch.nn.Parameter(torch.tensor([4.97e-2], dtype=torch.float32))

    def forward(self, T):
        T = torch.clamp(T, min=1e-6)
        term1 = self.a * T
        term2 = 1 - torch.exp(-self.b * T**2)
        term3 = 1.0
        term6 = torch.exp(-self.c * torch.sqrt(T))
        term4 = self.d * T**6
        term5 = self.e * T**4 * torch.exp(-self.f * T)
        return term1 * term2 * (term3 + term6 * term4 + term5)

# Instantiate models
model_adam = CpModel().to(device)
model_lm = CpModel().to(device)

print(f"Number of parameters: {sum(p.numel() for p in model_adam.parameters())}")

# === Wrap with training modules ===
module_adam = tlm.training.OptimizerModule(
    model=model_adam,
    optimizer=torch.optim.Adam(model_adam.parameters(), lr=1e-2),
    loss_fn=torch.nn.MSELoss(),
)

module_lm = tlm.training.LevenbergMarquardtModule(
    model=model_lm,
    loss_fn=tlm.loss.MSELoss(),
    learning_rate=1.0,
    solve_method="qr",
    attempts_per_step=10,
)

# === Train using Adam ===
print("\nTraining with Adam...")
start = time.perf_counter()
tlm.utils.fit(module_adam, loader, epochs=10)
end = time.perf_counter()
print(f"Adam training took {end - start:.2f} seconds")

# === Train using Levenberg-Marquardt ===
print("\nTraining with Levenberg-Marquardt...")
start = time.perf_counter()
tlm.utils.fit(module_lm, loader, epochs=10)
end = time.perf_counter()
print(f"LM training took {end - start:.2f} seconds")

# === Evaluate and Plot ===
print("\nPlotting results...")
with torch.no_grad():
    y_adam = model_adam(T_data)
    y_lm = model_lm(T_data)

T_np = T_data.cpu().numpy().flatten()
Cp_np = Cp_data.cpu().numpy().flatten()
y_adam_np = y_adam.cpu().numpy().flatten()
y_lm_np = y_lm.cpu().numpy().flatten()

output_notebook()
p = figure(title="Cp(T) Fit: Adam vs Levenberg-Marquardt", width=800, height=400)
p.circle(T_np, Cp_np, size=3, color="black", legend_label="Data")
p.line(T_np, y_adam_np, color="green", line_dash="dashed", line_width=2, legend_label="Adam")
p.line(T_np, y_lm_np, color="red", line_width=2, legend_label="LM")
p.legend.location = "top_left"
p.xaxis.axis_label = "T"
p.yaxis.axis_label = "Cp"
show(p)


Using device: cpu
Number of parameters: 6

Training with Adam...


Epoch 10/10: 100%|██████████| 10/10 [00:00<00:00, 293.75it/s, epoch_avg_loss=nan]


Training complete. Final epoch average loss: nan
Adam training took 0.39 seconds

Training with Levenberg-Marquardt...


Epoch 4/10:  10%|█         | 1/10 [00:00<00:00, 127.20it/s, epoch_avg_loss=3.6015e-07]

Training complete. Final epoch average loss: 3.6015e-07


LM training took 0.20 seconds

Plotting results...


In [7]:
def print_parameters(model):
    print("\nBest-fit parameters:")
    for name, param in model.named_parameters():
        print(f"{name} = {param.item():.8f}")

print_parameters(model_lm)



Best-fit parameters:
a = 0.00226886
b = 0.00028236
c = 4.48189735
d = 6.39312553
e = 0.00000001
f = 0.02022376


# equation 1 :

In [ ]:
def cp_custom(x0 , a=0.071187075 ,b=0.15814142 , c=547.8639 , d=3.90038959138114e-28 , e=6.626512e-10,f=4.2203817):
    numerator = x0 * (x0 * a + b)
    denominator = x0 + c / (d * x0**16 + x0**6 * e + x0 - f)
    return numerator / denominator

In [8]:
import time
import torch
import numpy as np
import pandas as pd
import torch_levenberg_marquardt as tlm
from torch.utils.data import TensorDataset
from bokeh.plotting import figure, show, output_notebook

# Set high matmul precision
torch.set_float32_matmul_precision('high')

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === Load your data from CSV ===
df = pd.read_csv("/content/magnesio-anthophyllite.csv")  # <-- update with your filename
T_data = torch.tensor(df["T"].values, dtype=torch.float32).unsqueeze(1).to(device)
Cp_data = torch.tensor(df["Cp"].values, dtype=torch.float32).unsqueeze(1).to(device)

# Create dataset and dataloader
dataset = TensorDataset(T_data, Cp_data)
loader = tlm.utils.FastDataLoader(dataset, batch_size=512, repeat=10, shuffle=True, device=device)

# === Define Cp model as a neural net with 6 trainable params ===
class CpModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.a = torch.nn.Parameter(torch.tensor([0.071187075], dtype=torch.float32))
        self.b = torch.nn.Parameter(torch.tensor([0.15814142], dtype=torch.float32))
        self.c = torch.nn.Parameter(torch.tensor([547.8639], dtype=torch.float32))
        self.d = torch.nn.Parameter(torch.tensor([3.90038959138114e-28], dtype=torch.float32))
        self.e = torch.nn.Parameter(torch.tensor([6.626512e-10], dtype=torch.float32))
        self.f = torch.nn.Parameter(torch.tensor([4.2203817], dtype=torch.float32))

    def forward(self, x0):
        # avoid division by zero
        x0 = torch.clamp(x0, min=1e-6)

        numerator = x0 * (x0 * self.a + self.b)
        denominator = x0 + self.c / (
            self.d * x0**16 + x0**6 * self.e + x0 - self.f
        )
        return numerator / denominator

# Instantiate models
model_adam = CpModel().to(device)
model_lm = CpModel().to(device)

print(f"Number of parameters: {sum(p.numel() for p in model_adam.parameters())}")

# === Wrap with training modules ===
module_adam = tlm.training.OptimizerModule(
    model=model_adam,
    optimizer=torch.optim.Adam(model_adam.parameters(), lr=1e-2),
    loss_fn=torch.nn.MSELoss(),
)

module_lm = tlm.training.LevenbergMarquardtModule(
    model=model_lm,
    loss_fn=tlm.loss.MSELoss(),
    learning_rate=1.0,
    solve_method="qr",
    attempts_per_step=10,
)

# === Train using Adam ===
print("\nTraining with Adam...")
start = time.perf_counter()
tlm.utils.fit(module_adam, loader, epochs=10)
end = time.perf_counter()
print(f"Adam training took {end - start:.2f} seconds")

# === Train using Levenberg-Marquardt ===
print("\nTraining with Levenberg-Marquardt...")
start = time.perf_counter()
tlm.utils.fit(module_lm, loader, epochs=10)
end = time.perf_counter()
print(f"LM training took {end - start:.2f} seconds")

# === Evaluate and Plot ===
print("\nPlotting results...")
with torch.no_grad():
    y_adam = model_adam(T_data)
    y_lm = model_lm(T_data)

T_np = T_data.cpu().numpy().flatten()
Cp_np = Cp_data.cpu().numpy().flatten()
y_adam_np = y_adam.cpu().numpy().flatten()
y_lm_np = y_lm.cpu().numpy().flatten()

output_notebook()
p = figure(title="Cp(T) Fit: Adam vs Levenberg-Marquardt", width=800, height=400)
p.circle(T_np, Cp_np, size=3, color="black", legend_label="Data")
p.line(T_np, y_adam_np, color="green", line_dash="dashed", line_width=2, legend_label="Adam")
p.line(T_np, y_lm_np, color="red", line_width=2, legend_label="LM")
p.legend.location = "top_left"
p.xaxis.axis_label = "T"
p.yaxis.axis_label = "Cp"
show(p)


Using device: cpu
Number of parameters: 6

Training with Adam...


Epoch 10/10: 100%|██████████| 10/10 [00:00<00:00, 282.53it/s, epoch_avg_loss=nan]


Training complete. Final epoch average loss: nan
Adam training took 0.45 seconds

Training with Levenberg-Marquardt...


Epoch 1/10:  20%|██        | 2/10 [00:00<00:00, 159.94it/s, epoch_avg_loss=3.9257e+01]


Training complete. Final epoch average loss: 3.9257e+01
LM training took 0.02 seconds

Plotting results...


# equation 2

In [11]:
import time
import torch
import numpy as np
import pandas as pd
import torch_levenberg_marquardt as tlm
from torch.utils.data import TensorDataset
from bokeh.plotting import figure, show, output_notebook

# Set high matmul precision
torch.set_float32_matmul_precision('high')

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === Load your data from CSV ===
df = pd.read_csv("/content/magnesio-anthophyllite.csv")  # <-- update with your filename
T_data = torch.tensor(df["T"].values, dtype=torch.float64).unsqueeze(1).to(device)
Cp_data = torch.tensor(df["Cp"].values, dtype=torch.float64).unsqueeze(1).to(device)

# Create dataset and dataloader
dataset = TensorDataset(T_data, Cp_data)
loader = tlm.utils.FastDataLoader(dataset, batch_size=512, repeat=10, shuffle=True, device=device)
torch.set_default_dtype(torch.float64)

class CpModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # Parameters for cp_base
        self.a = torch.nn.Parameter(torch.tensor([0.0071187075], dtype=torch.float64))
        self.b = torch.nn.Parameter(torch.tensor([0.15814142], dtype=torch.float64))
        self.c = torch.nn.Parameter(torch.tensor([547.8639], dtype=torch.float64))
        self.d = torch.nn.Parameter(torch.tensor([3.90038959138114e-28], dtype=torch.float64))
        self.e = torch.nn.Parameter(torch.tensor([6.626512e-10], dtype=torch.float64))
        self.f = torch.nn.Parameter(torch.tensor([4.2203817], dtype=torch.float64))

        # Parameters for residual_correction
        self.g = torch.nn.Parameter(torch.tensor([0.49417377], dtype=torch.float64))
        self.h = torch.nn.Parameter(torch.tensor([117.05849], dtype=torch.float64))
        self.i = torch.nn.Parameter(torch.tensor([0.066427015], dtype=torch.float64))
        self.j = torch.nn.Parameter(torch.tensor([0.17370676], dtype=torch.float64))
        self.k = torch.nn.Parameter(torch.tensor([2.855783], dtype=torch.float64))
        self.l = torch.nn.Parameter(torch.tensor([0.0018365078], dtype=torch.float64))
        self.m = torch.nn.Parameter(torch.tensor([1.0786958], dtype=torch.float64))

    def forward(self, x0):
        x0 = torch.clamp(x0, min=1e-6)
        x = x0 / 100  # optional normalization (recommended)

        # === cp_base ===
        numerator_base = x * (x * self.a + self.b)
        denominator_base = x + self.c / (
            self.d * x**16 + x**6 * self.e + x - self.f
        )
        cp_base = numerator_base / denominator_base

        # === residual_correction ===
        term_inner = -self.g - ((x - self.h) * -self.i) ** 2
        denominator_res = (x + term_inner ** 4) * self.j
        square_term = (self.k / denominator_res) ** 2
        residual = (square_term - self.l) * self.m

        return cp_base + residual

# Instantiate models
model_adam = CpModel().to(device)
model_lm = CpModel().to(device)

print(f"Number of parameters: {sum(p.numel() for p in model_adam.parameters())}")

# === Wrap with training modules ===
module_adam = tlm.training.OptimizerModule(
    model=model_adam,
    optimizer=torch.optim.Adam(model_adam.parameters(), lr=1e-2),
    loss_fn=torch.nn.MSELoss(),
)

module_lm = tlm.training.LevenbergMarquardtModule(
    model=model_lm,
    loss_fn=tlm.loss.MSELoss(),
    learning_rate=1.0,
    solve_method="qr",
    attempts_per_step=10,
)

# === Train using Adam ===
print("\nTraining with Adam...")
start = time.perf_counter()
tlm.utils.fit(module_adam, loader, epochs=10)
end = time.perf_counter()
print(f"Adam training took {end - start:.2f} seconds")

# === Train using Levenberg-Marquardt ===
print("\nTraining with Levenberg-Marquardt...")
start = time.perf_counter()
tlm.utils.fit(module_lm, loader, epochs=10)
end = time.perf_counter()
print(f"LM training took {end - start:.2f} seconds")

# === Evaluate and Plot ===
print("\nPlotting results...")
with torch.no_grad():
    y_adam = model_adam(T_data)
    y_lm = model_lm(T_data)

T_np = T_data.cpu().numpy().flatten()
Cp_np = Cp_data.cpu().numpy().flatten()
y_adam_np = y_adam.cpu().numpy().flatten()
y_lm_np = y_lm.cpu().numpy().flatten()

output_notebook()
p = figure(title="Cp(T) Fit: Adam vs Levenberg-Marquardt", width=800, height=400)
p.circle(T_np, Cp_np, size=3, color="black", legend_label="Data")
p.line(T_np, y_adam_np, color="green", line_dash="dashed", line_width=2, legend_label="Adam")
p.line(T_np, y_lm_np, color="red", line_width=2, legend_label="LM")
p.legend.location = "top_left"
p.xaxis.axis_label = "T"
p.yaxis.axis_label = "Cp"
show(p)


Using device: cpu
Number of parameters: 13

Training with Adam...


Epoch 10/10: 100%|██████████| 10/10 [00:00<00:00, 242.80it/s, epoch_avg_loss=1.1266e-02]


Training complete. Final epoch average loss: 1.1266e-02
Adam training took 0.48 seconds

Training with Levenberg-Marquardt...


Epoch 10/10: 100%|██████████| 10/10 [00:00<00:00, 127.85it/s, epoch_avg_loss=9.3079e-03]


Training complete. Final epoch average loss: 9.3079e-03
LM training took 0.85 seconds

Plotting results...


# equation 3 :